# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duashakeel0/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

The queue is built from the **client-holdout test set only** (1,979 rows, 6 clients Random Forest never saw in training) — the one population where the validated Precision@50=0.64 actually applies. Scoring the full dataset with a retrained model would produce a longer list, but I couldn't cite a validated precision number for rows the model has already seen.

**Reason codes** (one dominant reason per row, picked from the model's own top 3 features from Week 5/6 -- `avg_position_h1`, `impressions_h1`, `content_age_days`):
- `poor_position_declining_risk` -- ranked outside the top 20 (avg_position_h1 > 20)
- `high_exposure_declining_risk` -- meaningful impressions (>= 500) despite the risk flag
- `aging_content_declining_risk` -- older than the sample median age
- `model_flagged_declining_risk` -- flagged, but none of the above clearly dominant

**Action tiers**, from the model's probability:
- `prioritize_review` (probability >= 0.65) -- confidence: high
- `monitor` (0.40-0.65) -- confidence: medium
- `no_action_needed` (< 0.40) -- confidence: low

In [1]:
import os
import duckdb
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

token = os.environ["HF_TOKEN"]
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{token}')")

MONTH = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"

features_h1 = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS impressions_h1, SUM(gsc_clicks) AS clicks_h1,
        AVG(NULLIF(gsc_avg_position, 0)) AS avg_position_h1,
        SUM(ga4_sessions) AS sessions_h1, SUM(ga4_engaged_sessions) AS engaged_sessions_h1
    FROM {MONTH} WHERE report_date <= DATE '2026-03-15' AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()
target_h2 = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impressions_h2
    FROM {MONTH} WHERE report_date > DATE '2026-03-15' AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()
content_meta = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        DATE_DIFF('day', content_updated_date, DATE '2026-03-15') AS days_since_last_update,
        DATE_DIFF('day', content_created_date, DATE '2026-03-15') AS content_age_days,
        word_count
    FROM {CONTENT}
""").df()

data = (features_h1.merge(target_h2, on=["client_hash_id", "content_hash_id"], how="inner")
                    .merge(content_meta, on=["client_hash_id", "content_hash_id"], how="left"))
data = data[data["impressions_h1"] >= 50].copy()
data = data[(data["days_since_last_update"] >= 0) & (data["content_age_days"] >= 0)].copy()
data["is_declining_label"] = (data["impressions_h2"] < data["impressions_h1"]).astype(int)
data = data.fillna(0)
data = data.sort_values("content_hash_id").reset_index(drop=True)

FEATURES = ["impressions_h1", "clicks_h1", "avg_position_h1", "sessions_h1",
            "engaged_sessions_h1", "days_since_last_update", "content_age_days", "word_count"]

# Same client-holdout split as Week 5/6
clients = sorted(data["client_hash_id"].unique().tolist())
rng = np.random.default_rng(42)
rng.shuffle(clients)
n_test_clients = max(1, round(len(clients) * 0.2))
test_clients = set(clients[:n_test_clients])
train_clients = set(clients[n_test_clients:])
train_df = data[data["client_hash_id"].isin(train_clients)]
test_df = data[data["client_hash_id"].isin(test_clients)].copy()

rf = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42, class_weight="balanced").fit(
    train_df[FEATURES], train_df["is_declining_label"])
test_df["risk_score"] = rf.predict_proba(test_df[FEATURES])[:, 1]

# --- Reason codes: one dominant reason per row, from the model's own top features ---
median_age = data["content_age_days"].median()

def reason_code(row):
    if row["avg_position_h1"] > 20:
        return "poor_position_declining_risk"
    if row["impressions_h1"] >= 500:
        return "high_exposure_declining_risk"
    if row["content_age_days"] > median_age:
        return "aging_content_declining_risk"
    return "model_flagged_declining_risk"

def action_tier(score):
    if score >= 0.65:
        return "prioritize_review", "high"
    if score >= 0.40:
        return "monitor", "medium"
    return "no_action_needed", "low"

test_df["reason_code"] = test_df.apply(reason_code, axis=1)
tiers = test_df["risk_score"].apply(action_tier)
test_df["action"] = tiers.apply(lambda t: t[0])
test_df["confidence"] = tiers.apply(lambda t: t[1])

queue = test_df.sort_values("risk_score", ascending=False)[
    ["content_hash_id", "client_hash_id", "risk_score", "reason_code", "action", "confidence",
     "impressions_h1", "avg_position_h1", "content_age_days"]
].reset_index(drop=True)

print(f"Queue built from {len(queue):,} held-out rows (validated Precision@50 = 0.64 applies to this exact population)")
print("\nAction distribution:")
print(queue["action"].value_counts())
print("\nReason code distribution among prioritize_review rows:")
print(queue[queue["action"] == "prioritize_review"]["reason_code"].value_counts())
print("\nTop 10 of the queue:")
print(queue.head(10).to_string(index=False))

Queue built from 1,979 held-out rows (validated Precision@50 = 0.64 applies to this exact population)

Action distribution:
action
monitor              1763
no_action_needed      205
prioritize_review      11
Name: count, dtype: int64

Reason code distribution among prioritize_review rows:
reason_code
high_exposure_declining_risk    9
poor_position_declining_risk    2
Name: count, dtype: int64

Top 10 of the queue:
         content_hash_id          client_hash_id  risk_score                  reason_code            action confidence  impressions_h1  avg_position_h1  content_age_days
content_66d1fffc91f4f029 client_20259bd6705d81d4    0.775827 high_exposure_declining_risk prioritize_review       high         15096.0        12.638678               145
content_283e87bc4e224d58 client_20259bd6705d81d4    0.770606 high_exposure_declining_risk prioritize_review       high          9521.0        17.567028               145
content_097459d155cccb26 client_20259bd6705d81d4    0.759261 high_expos

## 2. Intended use and limits

**Who uses this:** a content editor or SEO strategist with limited weekly review capacity, deciding which pages to look at first (the exact decision framed back in Week 1).

**What it's for:** ranking candidates for human review, using a validated (Precision@50 = 0.64 on held-out clients) but imperfect signal -- not a verdict on any single page.

**Where it stops being valid:**
- **One month, one snapshot.** Trained and tested on March 2026 only. Seasonal content, or a portfolio with a different rhythm, may not behave the same way.
- **6 held-out clients.** Per Week 6, that's a small test population -- treat the 0.64 number as "this run's evidence," not a guaranteed future rate.
- **Not causal.** A `prioritize_review` tag means "worth a look," never "refreshing this will fix it" -- no experiment was run (Week 1's careful-words commitment, still holding).
- **Population gap, named in Week 6:** pages edited between March 15 and July aren't in this queue at all (Week 5's leakage fix excluded them). If your true priority page was already touched by an editor in that window, it won't appear here.
- **Client-specific drift not modeled.** A client whose site changed significantly (redesign, migration) after March isn't represented.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

**What a person must check before acting on any `prioritize_review` row** (from the lane guide's decline-vs-lookalike checklist, Section 7):
- **Consolidation:** did a sibling page on the same site absorb the demand? Check related-content groups before assuming decline.
- **Seasonality:** does this topic naturally dip on a calendar? Compare to the same period last year if history exists.
- **Noise:** is `impressions_h1` large enough that the drop isn't just normal volume wobble?
- **Position sanity:** does `avg_position_h1` for this row make sense for the reason code assigned? (E.g. a `poor_position_declining_risk` row should actually show a weak position -- verify it does, not just trust the label.)

**What should never be automated:**
- Never auto-publish a content change based on this queue -- every action here ends with a human decision, not a script.
- Never treat `prioritize_review` as proof the page is declining -- it's a ranking signal with 0.64 precision, meaning roughly 1 in 3 flagged pages in this run were not actually declining.
- Never use this queue to justify removing a page, deprioritizing a client relationship, or any action beyond "look at this one sooner."
- Never skip the consolidation/seasonality/noise check above because the model score is high -- a high score does not substitute for that read.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

What would tell me the recommendations went stale:

- **Precision drift:** re-run Precision@50 on a fresh month's held-out clients; if it falls meaningfully below ~0.5 (roughly halfway between the 0.64 measured here and the 0.356 base rate), the ranking is losing its edge over guessing.
- **Base-rate shift:** if the declining-rate base rate moves far from ~0.35-0.55 (the range seen across Weeks 5-7), the underlying content mix has changed enough that thresholds tuned here may no longer fit.
- **New/departed clients:** onboarding clients not represented in the training clients, or losing several that were, changes what "generalizes" even means -- retrain when the client roster shifts materially.
- **Feature importance reshuffle:** if a fresh run's top feature stops being `avg_position_h1`/`impressions_h1`/`content_age_days` in that rough order, something about the portfolio changed and the reason-code logic (tied to those specific features) needs re-deriving, not just relabeling.
- **Calendar cadence:** at minimum, re-validate monthly -- this was built and tested on one month (March), and the lane guide's own numbers show client history and behavior are not stationary across months.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

Writes the ranked queue (regenerable, gitignored by design) and one figure the capstone paper reuses: the action-tier breakdown with its reason-code mix.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

os.makedirs("../outputs", exist_ok=True)
os.makedirs("../outputs/figures", exist_ok=True)

queue.to_csv("../outputs/action_playbook_queue.csv", index=False)
print(f"Wrote ../outputs/action_playbook_queue.csv ({len(queue):,} rows)")

# Theme-safe styling: transparent background, mid-luminance text/grid that
# reads on both a light and dark card on the deployed paper page.
mpl.rcParams.update({
    "figure.facecolor": "none", "axes.facecolor": "none", "savefig.facecolor": "none",
    "text.color": "#7a7d74", "axes.labelcolor": "#7a7d74",
    "xtick.color": "#7a7d74", "ytick.color": "#7a7d74",
    "axes.edgecolor": "#7a7d74", "grid.color": "#7a7d74", "grid.alpha": 0.15,
    "font.family": "serif", "font.size": 11.5,
})

fig, ax = plt.subplots(figsize=(6.4, 4.2))
counts = queue["action"].value_counts().reindex(["prioritize_review", "monitor", "no_action_needed"]).fillna(0)
colors = {"prioritize_review": "#b8532e", "monitor": "#c9a227", "no_action_needed": "#3a9b7d"}
bars = ax.bar([a.replace("_", " ") for a in counts.index], counts.values,
              color=[colors[a] for a in counts.index], width=0.55, zorder=3)
ax.set_ylabel("Content items")
ax.set_title("Action playbook: tier breakdown\n(held-out client test set, n=%d)" % len(queue), fontsize=12)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", zorder=0)
for b, v in zip(bars, counts.values):
    ax.text(b.get_x() + b.get_width()/2, v + max(counts.values) * 0.02, f"{int(v)}", ha="center", fontsize=12, fontweight="bold", color="#3d403a")
plt.tight_layout()
fig.savefig("../outputs/figures/w07_action_tier_breakdown.png", dpi=180, transparent=True)
plt.close(fig)
print("Wrote ../outputs/figures/w07_action_tier_breakdown.png")

summary = {
    "total_rows": int(len(queue)),
    "action_counts": counts.astype(int).to_dict(),
    "prioritize_review_reason_codes": queue[queue["action"] == "prioritize_review"]["reason_code"].value_counts().to_dict(),
    "validated_precision_at_50": 0.64,
    "base_rate": round(float(test_df["is_declining_label"].mean()), 3),
}
print("\nSummary for the paper's Results section:")
for k, v in summary.items():
    print(f"  {k}: {v}")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.